In [13]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Unzip your data
!unzip /content/drive/MyDrive/defect_data.zip -d /content/defect-detection

# Install YOLOv8
!pip install ultralytics

# Train
import os
os.chdir('/content/drive/MyDrive/defect-detection')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
unzip:  cannot find or open /content/drive/MyDrive/defect_data.zip, /content/drive/MyDrive/defect_data.zip.zip or /content/drive/MyDrive/defect_data.zip.ZIP.


In [3]:
import os
labels = os.listdir('/content/drive/MyDrive/defect-detection/data/steel/labels/train')
print(labels[:5])

['inclusion_86.txt', 'inclusion_44.txt', 'inclusion_89.txt', 'inclusion_65.txt', 'inclusion_56.txt']


In [4]:
import os
print(os.listdir('/content/drive/MyDrive/defect-detection/data/steel/labels/train')[:5])

['inclusion_86.txt', 'inclusion_44.txt', 'inclusion_89.txt', 'inclusion_65.txt', 'inclusion_56.txt']


In [5]:
import os

# Check if labels exist
label_path = '/content/drive/MyDrive/defect-detection/data/steel/labels/train'
files = os.listdir(label_path)
print(f"Total label files: {len(files)}")
print("Sample files:", files[:5])

# Check content of one label file
with open(os.path.join(label_path, files[0])) as f:
    print("Content:", f.read())

Total label files: 1439
Sample files: ['inclusion_86.txt', 'inclusion_44.txt', 'inclusion_89.txt', 'inclusion_65.txt', 'inclusion_56.txt']
Content: 1 0.237500 0.215000 0.125000 0.420000
1 0.132500 0.345000 0.105000 0.260000
1 0.087500 0.712500 0.105000 0.325000
1 0.267500 0.835000 0.185000 0.250000
1 0.635000 0.327500 0.170000 0.445000



In [7]:
import os

# Check images
train_images = os.listdir('/content/drive/MyDrive/defect-detection/data/steel/images/train')
print("Sample images:", train_images[:3])

# Check labels
train_labels = os.listdir('/content/drive/MyDrive/defect-detection/data/steel/labels/train')
print("Sample labels:", train_labels[:3])

# Check if names match
img_name = train_images[0].replace('.jpg', '.txt').replace('.png', '.txt')
print(f"\nLooking for: {img_name}")
print(f"Exists in labels: {img_name in train_labels}")

Sample images: ['images']
Sample labels: ['inclusion_86.txt', 'inclusion_44.txt', 'inclusion_89.txt']

Looking for: images
Exists in labels: False


In [8]:
import shutil
import os

src = '/content/drive/MyDrive/defect-detection/data/steel/images/train/images'
dst = '/content/drive/MyDrive/defect-detection/data/steel/images/train'

# Move all files up one level
for file in os.listdir(src):
    shutil.move(os.path.join(src, file), os.path.join(dst, file))

# Remove empty folder
os.rmdir(src)
print("Done ✅")

# Do the same for val
src_val = '/content/drive/MyDrive/defect-detection/data/steel/images/val/images'
dst_val = '/content/drive/MyDrive/defect-detection/data/steel/images/val'

for file in os.listdir(src_val):
    shutil.move(os.path.join(src_val, file), os.path.join(dst_val, file))

os.rmdir(src_val)
print("Val done ✅")

Done ✅
Val done ✅


In [10]:
import os

# Check exact contents of images/train
train_path = '/content/drive/MyDrive/defect-detection/data/steel/images/train'
contents = os.listdir(train_path)
print("Images train contents:", contents[:5])
print("Total:", len(contents))

# Check one image name vs one label name
label_path = '/content/drive/MyDrive/defect-detection/data/steel/labels/train'
labels = os.listdir(label_path)
print("\nLabel sample:", labels[0])
print("Image sample:", contents[0])

# Check if they match
img_stem = contents[0].replace('.jpg','').replace('.png','').replace('.bmp','')
lbl_stem = labels[0].replace('.txt','')
print(f"\nImage stem: {img_stem}")
print(f"Label stem: {lbl_stem}")
print(f"Match possible: {img_stem == lbl_stem}")

Images train contents: ['patches', 'rolled-in_scale', 'pitted_surface', 'scratches', 'inclusion']
Total: 6

Label sample: inclusion_86.txt
Image sample: patches

Image stem: patches
Label stem: inclusion_86
Match possible: False


In [11]:
import shutil
import os

# Move all images from subfolders to train folder directly
train_path = '/content/drive/MyDrive/defect-detection/data/steel/images/train'

for class_folder in os.listdir(train_path):
    class_path = os.path.join(train_path, class_folder)
    if os.path.isdir(class_path):
        for img_file in os.listdir(class_path):
            shutil.move(os.path.join(class_path, img_file), os.path.join(train_path, img_file))
        os.rmdir(class_path)
        print(f"Moved {class_folder} ✅")

# Do same for val
val_path = '/content/drive/MyDrive/defect-detection/data/steel/images/val'

for class_folder in os.listdir(val_path):
    class_path = os.path.join(val_path, class_folder)
    if os.path.isdir(class_path):
        for img_file in os.listdir(class_path):
            shutil.move(os.path.join(class_path, img_file), os.path.join(val_path, img_file))
        os.rmdir(class_path)
        print(f"Moved val/{class_folder} ✅")

print("\nAll done ✅")

Moved patches ✅
Moved rolled-in_scale ✅
Moved pitted_surface ✅
Moved scratches ✅
Moved inclusion ✅
Moved crazing ✅
Moved val/scratches ✅
Moved val/rolled-in_scale ✅
Moved val/pitted_surface ✅
Moved val/patches ✅
Moved val/crazing ✅
Moved val/inclusion ✅

All done ✅


In [12]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data='/content/drive/MyDrive/defect-detection/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    name='steel_defect_v1'
)


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/defect-detection/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=steel_defect_v16, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=

In [14]:
from google.colab import files
files.download('/content/drive/MyDrive/defect-detection/runs/detect/steel_defect_v16/weights/best.pt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>